# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished} | License: {metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All Croissant entities are referenced by their `@id`.

In [ ]:
# List and inspect available record sets and their fields (by @id)
record_sets = list(dataset.record_sets)
print(f"Available record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set name: {rs.name} (ID: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (ID: {field.id}) type={field.data_type} column@id={field.column.id if field.column else None}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all @id for record sets
record_set_ids = [r.id for r in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Choose the main table for further analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Main table: {main_record_set_id}")
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No records found in any record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's:
- Pick a numeric field based on the data overview.
- Filter, normalize, and group as an example.

In [ ]:
# Identify a numeric field and a group field
import numpy as np

df = dataframes[main_record_set_id]

# Try to guess a likely numeric field (e.g. 'cr:peptide_count' or 'cr:coverage_percent' or similar)
possible_numeric_fields = [col for col in df.columns if df[col].dtype in [np.number, float, int] or np.issubdtype(df[col].dtype, np.number)]
if not possible_numeric_fields:
    # Try columns with numeric-looking names
    possible_numeric_fields = [col for col in df.columns if ('count' in col.lower() or 'percent' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower() or 'pi' in col.lower())]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    print('No obvious numeric field found, please select manually.')
    numeric_field = df.columns[0]

threshold = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0

# Filtering
condition = df[numeric_field] > threshold if np.issubdtype(df[numeric_field].dtype, np.number) else pd.Series([True] * len(df))
filtered_df = df[condition].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalizing
if np.issubdtype(filtered_df[numeric_field].dtype, np.number):
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / (filtered_df[numeric_field].std() + 1e-8)
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"{numeric_field} is not numeric; skipping normalization.")

# Grouping by a likely categorical field (e.g. sample or category)
group_field_candidates = [col for col in df.columns if col != numeric_field and (df[col].dtype == object or 'sample' in col.lower() or 'category' in col.lower() or 'class' in col.lower())]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field and group_field in filtered_df.columns and np.issubdtype(filtered_df[numeric_field].dtype, np.number):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print('No valid group field found or numeric field not suitable for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram of the selected numeric field
if np.issubdtype(df[numeric_field].dtype, np.number):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

# If grouped, show bar plot
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10,5))
    grouped_df.sort_values(ascending=False).plot(kind='bar')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides mass spectrometry-based protein abundance and characteristics from human samples.
- The loaded data contains fields such as protein accession, peptide counts, molecular weight, pI, and normalized abundances.
- Simple EDA reveals the distribution of key numeric properties and enables filtering/grouping for further analysis.

To extend this analysis, refine field/column selection based on biological context and integrate with downstream bioinformatics pipelines.